# HVAC IoT Telemetry Time-Series Analysis

## Objective

This notebook analyzes telemetry data collected from HVAC IoT devices installed across multiple store locations.

The analysis includes:

- Data cleaning
- Time-series preparation
- Exploratory Data Analysis
- Anomaly detection
- Device performance comparison
- Business insights

In [490]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [491]:
df_raw = pd.read_csv("/content/device_telemetry.csv")
df = df_raw.copy()

In [492]:
df.head()

,device_id,timestamp,room_temp_c,setpoint_c,ambient_temp_c,compressor_state,power_watt,mode
0,DVC-103,2026-06-06 03:15:00,25.74,26.0,25.74,0,49.0,OFF
1,DVC-102,2026-06-14 20:15:00,23.96,24.0,31.83,1,1420.6,COOL
2,DVC-101,2026-06-14 16:45:00,24.56,24.0,37.48,0,42.6,FAN
3,DVC-103,2026-06-01 03:30:00,26.04,26.0,25.88,0,52.1,OFF
4,DVC-102,2026-06-13 19:00:00,24.44,24.0,33.47,0,17.5,FAN


In [493]:
print("Dataset Shape:", df.shape)

Dataset Shape: (6470, 8)


In [494]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6470 entries, 0 to 6469
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   device_id         6470 non-null   object 
 1   timestamp         6470 non-null   object 
 2   room_temp_c       6470 non-null   float64
 3   setpoint_c        6470 non-null   float64
 4   ambient_temp_c    6470 non-null   float64
 5   compressor_state  6470 non-null   int64  
 6   power_watt        6470 non-null   float64
 7   mode              6470 non-null   object 
dtypes: float64(4), int64(1), object(3)
memory usage: 404.5+ KB


In [495]:
df.describe()

,room_temp_c,setpoint_c,ambient_temp_c,compressor_state,power_watt
count,6470.000000,6470.000000,6470.000000,6470.000000,6470.000000
mean,24.907362,25.002782,30.980975,0.352396,515.042875
std,5.240202,1.000073,4.289170,0.477753,655.594466
min,-99.000000,24.000000,23.050000,0.000000,8.000000
25%,24.310000,24.000000,26.802500,0.000000,42.425000
50%,24.795000,26.000000,30.980000,0.000000,52.800000
75%,25.880000,26.000000,35.190000,1.000000,1285.975000
max,85.000000,26.000000,38.740000,1.000000,5024.600000


In [496]:
print(df.isnull().sum())

device_id           0
timestamp           0
room_temp_c         0
setpoint_c          0
ambient_temp_c      0
compressor_state    0
power_watt          0
mode                0
dtype: int64


In [497]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 32


In [498]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(df["timestamp"].dtype)

datetime64[ns]


In [499]:
print("Duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicate rows after cleaning:", df.duplicated().sum())

Duplicate rows before cleaning: 32
Duplicate rows after cleaning: 0


In [500]:
df = df.sort_values(by=["device_id", "timestamp"]).reset_index(drop=True)
df.head()

,device_id,timestamp,room_temp_c,setpoint_c,ambient_temp_c,compressor_state,power_watt,mode
0,DVC-101,2026-06-01 00:00:00,26.89,26.0,26.94,0,68.1,OFF
1,DVC-101,2026-06-01 00:15:00,26.89,26.0,25.86,0,37.7,OFF
2,DVC-101,2026-06-01 00:30:00,26.97,26.0,26.69,0,43.9,OFF
3,DVC-101,2026-06-01 00:45:00,26.93,26.0,26.58,0,50.6,OFF
4,DVC-101,2026-06-01 01:00:00,26.71,26.0,24.63,0,48.5,OFF


In [501]:
print("Dataset Shape:", df.shape)
df.info()

Dataset Shape: (6438, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6438 entries, 0 to 6437
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   device_id         6438 non-null   object        
 1   timestamp         6438 non-null   datetime64[ns]
 2   room_temp_c       6438 non-null   float64       
 3   setpoint_c        6438 non-null   float64       
 4   ambient_temp_c    6438 non-null   float64       
 5   compressor_state  6438 non-null   int64         
 6   power_watt        6438 non-null   float64       
 7   mode              6438 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 402.5+ KB


In [502]:
resampled = []
for device in df["device_id"].unique():
    temp = (
        df[df["device_id"] == device]
        .set_index("timestamp")
        .resample("15min")
        .first()
    )
    temp["device_id"] = device
    resampled.append(temp)
df = (
    pd.concat(resampled)
      .reset_index()
)

In [503]:
df["Offline"] = df["room_temp_c"].isna()

In [504]:
print(df.shape)
print(df["Offline"].value_counts())

(6720, 9)
Offline
False    6438
True      282
Name: count, dtype: int64


In [505]:
offline_summary = (
    df.groupby("device_id")["Offline"]
      .sum()
      .reset_index(name="Offline_Intervals")
)

offline_summary

,device_id,Offline_Intervals
0,DVC-101,22
1,DVC-102,31
2,DVC-103,174
3,DVC-104,31
4,DVC-105,24


In [506]:
print(df["room_temp_c"].describe())

count    6438.00000
mean       24.90634
std         5.25274
min       -99.00000
25%        24.31000
50%        24.79000
75%        25.88000
max        85.00000
Name: room_temp_c, dtype: float64


In [507]:
invalid_temp = df[
    (df["room_temp_c"] < 10) |
    (df["room_temp_c"] > 40)]
print("Invalid Room Temperature Readings:", len(invalid_temp))
invalid_temp[[
    "device_id",
    "timestamp",
    "room_temp_c"
]].head(20)

Invalid Room Temperature Readings: 25


,device_id,timestamp,room_temp_c
141,DVC-101,2026-06-02 11:15:00,-99.0
157,DVC-101,2026-06-02 15:15:00,85.0
383,DVC-101,2026-06-04 23:45:00,0.0
398,DVC-101,2026-06-05 03:30:00,-99.0
496,DVC-101,2026-06-06 04:00:00,85.0
512,DVC-101,2026-06-06 08:00:00,-99.0
763,DVC-101,2026-06-08 22:45:00,-99.0
1091,DVC-101,2026-06-12 08:45:00,0.0
1154,DVC-101,2026-06-13 00:30:00,0.0
1160,DVC-101,2026-06-13 02:00:00,85.0


In [508]:
print(df["power_watt"].describe())

count    6438.000000
mean      515.722103
std       655.883369
min         8.000000
25%        42.500000
50%        52.900000
75%      1285.975000
max      5024.600000
Name: power_watt, dtype: float64


In [509]:
print(df["compressor_state"].value_counts())

compressor_state
0.0    4166
1.0    2272
Name: count, dtype: int64


In [510]:
print(df["mode"].value_counts())

mode
OFF     2693
COOL    2272
FAN     1473
Name: count, dtype: int64


In [511]:
print(df.isnull().sum())

timestamp             0
device_id             0
room_temp_c         282
setpoint_c          282
ambient_temp_c      282
compressor_state    282
power_watt          282
mode                282
Offline               0
dtype: int64


## Handling Invalid Sensor Readings

The data quality assessment identified a small number of unrealistic room temperature readings (e.g., -99°C, 0°C and 85°C). These values are treated as sensor errors because they fall outside the normal operating range of an HVAC system.

To prepare the dataset for analysis, only these invalid sensor readings are corrected. Genuine offline periods introduced during resampling are preserved as missing values to accurately represent telemetry outages.

In [512]:
offline_mask = df["Offline"].copy()

In [513]:
invalid_mask = (
    (df["room_temp_c"] < 10) |
    (df["room_temp_c"] > 40)
)

print("Invalid readings:", invalid_mask.sum())

Invalid readings: 25


In [514]:
df.loc[invalid_mask, "room_temp_c"] = np.nan

In [515]:
df["room_temp_c"] = (
    df.groupby("device_id")["room_temp_c"]
      .transform(lambda x: x.ffill().bfill())
)

In [516]:
df.loc[offline_mask, "room_temp_c"] = np.nan

In [517]:
print("Remaining invalid temperatures:")
print(((df["room_temp_c"] < 10) | (df["room_temp_c"] > 40)).sum())

print("\nOffline periods:")
print(df["Offline"].sum())

print("\nMissing room temperatures:")
print(df["room_temp_c"].isna().sum())

Remaining invalid temperatures:
0

Offline periods:
282

Missing room temperatures:
282


# Data preparation Summary & Observations

- Successfully loaded and explored the HVAC telemetry dataset to understand its structure, data types, and overall quality.
- Converted the `timestamp` column to the correct datetime format to enable time-series analysis.
- Removed duplicate records to improve data consistency and avoid redundant observations.
- Sorted the dataset by device ID and timestamp to maintain chronological order for each HVAC device.
- Resampled the telemetry data to a regular 15-minute interval, creating a consistent time series across all devices.
- Identified and flagged missing telemetry intervals as **offline periods**, preserving communication gaps for later analysis.
- Performed a comprehensive data quality assessment to identify unrealistic sensor readings and validate key variables.
- Detected a small number of invalid room temperature values (e.g., -99°C, 0°C, and 85°C) that were outside the expected HVAC operating range.
- Corrected only the invalid sensor readings using neighbouring valid observations while preserving genuine offline periods as missing values.
- Produced a clean, consistent, and analysis-ready dataset suitable for time-series analysis, visualization, energy estimation, and anomaly detection.

---



#  Time-Series Analysis

This section analyzes the temporal behavior of the HVAC devices using hourly and daily summaries. The analysis focuses on room temperature, power consumption, and compressor runtime to understand operational patterns, identify anomalies, and evaluate device performance over time.

In [518]:
df["runtime_hours"] = df["compressor_state"] * 0.25

In [519]:
hourly_summary = (
    df.groupby(
        [
            "device_id",
            pd.Grouper(key="timestamp", freq="H")
        ]
    )
    .agg(avg_room_temp=("room_temp_c", "mean"),
         avg_power=("power_watt", "mean"),
        compressor_runtime=("runtime_hours", "sum")
    )
    .reset_index())

hourly_summary.head()

/tmp/ipykernel_916/3859700324.py:5: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



,device_id,timestamp,avg_room_temp,avg_power,compressor_runtime
0,DVC-101,2026-06-01 00:00:00,26.920000,50.075,0.0
1,DVC-101,2026-06-01 01:00:00,26.722500,45.150,0.0
2,DVC-101,2026-06-01 02:00:00,26.343333,52.200,0.0
3,DVC-101,2026-06-01 03:00:00,26.022500,1240.075,0.0
4,DVC-101,2026-06-01 04:00:00,26.012500,41.550,0.0


In [520]:
print(hourly_summary.shape)
hourly_summary.info()

(1680, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1680 entries, 0 to 1679
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   device_id           1680 non-null   object        
 1   timestamp           1680 non-null   datetime64[ns]
 2   avg_room_temp       1644 non-null   float64       
 3   avg_power           1644 non-null   float64       
 4   compressor_runtime  1680 non-null   float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 65.8+ KB


In [521]:
hourly_summary.describe()

,timestamp,avg_room_temp,avg_power,compressor_runtime
count,1680,1644.000000,1644.000000,1680.000000
mean,2026-06-07 23:30:00.000000256,25.037507,514.711273,0.338095
min,2026-06-01 00:00:00,23.500000,25.175000,0.000000
25%,2026-06-04 11:45:00,24.360000,45.950000,0.000000
50%,2026-06-07 23:30:00,24.658750,639.787500,0.375000
75%,2026-06-11 11:15:00,25.865625,974.100000,0.750000
max,2026-06-14 23:00:00,27.313333,2228.500000,1.000000
std,NaN,0.908726,471.467602,0.349604


In [522]:
daily_summary = (
    df.groupby(
        [
            "device_id",
            pd.Grouper(key="timestamp", freq="D")
        ]
    )
    .agg(
        avg_room_temp=("room_temp_c", "mean"),
        avg_power=("power_watt", "mean"),
        compressor_runtime=("runtime_hours", "sum")
    )
    .reset_index())

daily_summary.head()

,device_id,timestamp,avg_room_temp,avg_power,compressor_runtime
0,DVC-101,2026-06-01,25.317097,607.659140,8.50
1,DVC-101,2026-06-02,24.982316,524.303158,8.00
2,DVC-101,2026-06-03,24.981053,516.132632,8.00
3,DVC-101,2026-06-04,24.920645,504.249462,8.25
4,DVC-101,2026-06-05,25.236632,534.616842,8.00


In [523]:
print(daily_summary.shape)
daily_summary.info()

(70, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   device_id           70 non-null     object        
 1   timestamp           70 non-null     datetime64[ns]
 2   avg_room_temp       70 non-null     float64       
 3   avg_power           70 non-null     float64       
 4   compressor_runtime  70 non-null     float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 2.9+ KB


In [524]:
daily_summary.describe()

,timestamp,avg_room_temp,avg_power,compressor_runtime
count,70,70.000000,70.000000,70.000000
mean,2026-06-07 12:00:00,25.039683,509.551871,8.114286
min,2026-06-01 00:00:00,23.500000,42.545833,0.000000
25%,2026-06-04 00:00:00,25.065931,497.994903,8.000000
50%,2026-06-07 12:00:00,25.151325,511.711383,8.375000
75%,2026-06-11 00:00:00,25.248732,534.282233,8.500000
max,2026-06-14 00:00:00,25.549167,607.659140,9.250000
std,NaN,0.492298,64.717687,1.315794


## Summary & Observations

- Hourly and daily summaries were successfully generated for each HVAC device using the cleaned telemetry data.
- The summaries include key operational metrics such as average room temperature, average power consumption, and compressor runtime, providing both short-term and long-term views of device performance.
- Hourly aggregation captures variations in HVAC operation throughout the day, making it useful for identifying changes in operating behavior and periods of high or low activity.
- Daily aggregation smooths short-term fluctuations and highlights overall performance trends, enabling easier comparison of device operation across multiple days.
- Compressor runtime serves as an indicator of HVAC workload, while average power consumption provides insight into the energy demand of each device.
- These aggregated datasets establish a strong foundation for the subsequent visualizations, anomaly detection, energy consumption estimation, and device performance analysis.

In [525]:
fig = px.line(
    df,
    x="timestamp",
    y="room_temp_c",
    color="device_id",
    title="Room Temperature Trend by Device",
    labels={
        "timestamp": "Time",
        "room_temp_c": "Room Temperature (°C)",
        "device_id": "Device"
    },
    render_mode="svg")

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    height=550,
    legend_title="HVAC Device",
    title_x=0.5)

fig.show()
fig.write_html(
    "01_room_temperature.html",
    full_html=False,
    include_plotlyjs=True
)

#### Observations

- All five HVAC devices exhibit broadly similar temperature patterns, indicating consistent operation under comparable environmental conditions.
- Room temperatures generally fluctuate between **24°C and 27°C**, suggesting that the HVAC systems maintain temperatures within an acceptable operating range.
- The periodic rise and fall in temperature reflects normal compressor cycling as the devices regulate room temperature around the desired level.
- **DVC-104 displays a prolonged flat-line temperature of approximately 23.5°C towards the end of the observation period.** This behaviour is unusual compared to the other devices and may indicate a sensor issue, prolonged steady-state operation, or a telemetry-related anomaly. This device will be examined further during the anomaly detection stage.
- Overall, no significant temperature spikes or extreme fluctuations are observed, indicating that the HVAC systems generally operate within expected temperature limits.

In [526]:
fig = px.line(
    df,
    x="timestamp",
    y=["room_temp_c", "setpoint_c"],
    facet_row="device_id",
    title="Room Temperature vs Setpoint",
    labels={
        "value": "Temperature (°C)",
        "timestamp": "Time"
    },
    render_mode="svg")
fig.update_layout(
    template="plotly_white",
    height=1000,
    title_x=0.5,
    hovermode="x unified"
)

fig.show()
fig.write_html(
    "02_room_vs_setpoint.html",
    full_html=False,
    include_plotlyjs=True
)

#### Key Observations

- For most HVAC devices, the room temperature closely follows the configured setpoint, indicating effective temperature regulation during normal operation.
- Small deviations from the setpoint are observed across all devices, reflecting the normal compressor cycling and the thermal response of the conditioned space.
- Room temperatures generally return toward the target setpoint after each operating cycle, suggesting that the HVAC control systems are functioning as expected.
- **DVC-103 exhibits a temporary gap in telemetry**, indicating a period where no sensor readings were available. This may represent a communication interruption or temporary device downtime.
- **DVC-104 displays an extended flat-line temperature of approximately 23.5°C while the setpoint continues to change.** This behaviour is significantly different from the other devices and suggests a potential sensor malfunction, communication issue, or operational anomaly. This device will be investigated further during the anomaly detection stage.
- Overall, the visualization indicates that the majority of HVAC devices successfully maintain temperatures close to their configured setpoints, with only minor short-term variations that are expected during normal operation.

In [527]:
fig = px.line(
    df,
    x="timestamp",
    y="power_watt",
    color="device_id",
    title="Power Consumption Trend by Device",
    labels={
        "timestamp": "Time",
        "power_watt": "Power Consumption (W)",
        "device_id": "HVAC Device"
    },
    render_mode="svg"
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    height=550,
    title_x=0.5,
    legend_title="HVAC Device"
)

fig.show()
fig.write_html(
    "03_power_trend.html",
    full_html=False,
    include_plotlyjs=True
)

In [528]:
fig = px.line(
    daily_summary,
    x="timestamp",
    y="avg_power",
    color="device_id",
    title="Daily Average Power Consumption",
    labels={
        "timestamp": "Date",
        "avg_power": "Average Power (W)",
        "device_id": "HVAC Device"
    },
    markers=True
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    height=550,
    title_x=0.5,
    legend_title="HVAC Device"
)

fig.show()
fig.write_html(
    "04_daily_power.html",
    full_html=False,
    include_plotlyjs=True
)

#### Key Observations

- The daily average power consumption for most HVAC devices remains relatively stable, generally ranging between **480 W and 600 W**, indicating consistent day-to-day operation.
- **DVC-101** records the highest average daily power consumption on several days, suggesting that it operated under a relatively higher workload than the other devices.
- **DVC-103 exhibits a sharp drop in average power consumption on 7 June**, followed by a gradual recovery over the next few days. This behaviour is significantly different from the other devices and may indicate a temporary device outage, communication interruption, or operational anomaly.
- **DVC-102, DVC-104, and DVC-105** maintain fairly consistent daily power consumption throughout the observation period, with only minor fluctuations that are expected during normal HVAC operation.
- Overall, the daily power trends indicate stable energy usage across most devices, while **DVC-103 requires further investigation due to its abnormal reduction in power consumption.**

In [529]:
fig = px.line(
    hourly_pattern,
    x="hour",
    y="avg_power",
    markers=True,
    title="Average Hourly Power Consumption (Peak vs Off-Peak)",
    labels={
        "hour": "Hour of Day",
        "avg_power": "Average Power (W)"
    }
)

fig.update_layout(
    template="plotly_white",
    height=500,
    title_x=0.5
)

fig.show()
fig.write_html(
    "05_peak_offpeak.html",
    full_html=False,
    include_plotlyjs=True
)

#### Key Observations

- HVAC power consumption follows a clear daily operating pattern, with distinct peak and off-peak periods.
- **Off-peak usage** occurs during the late evening, night, and early morning hours (approximately **21:00–07:00**), where the average power consumption remains below **100 W**, indicating minimal HVAC activity.
- Power consumption increases sharply after **08:00**, reaching its highest level at approximately **09:00**, where the average power demand peaks at around **1,380 W**.
- Between **10:00 and 20:00**, power consumption remains relatively high (approximately **750–900 W**), indicating sustained HVAC operation during normal business or occupied hours.
- The observed pattern suggests that HVAC systems are primarily active during daytime operating hours and significantly reduce energy consumption during periods of lower occupancy, demonstrating an expected daily energy usage profile.

## Anomaly Detection

In [530]:
offline_summary = (
    df.groupby("device_id")["Offline"]
      .sum()
      .reset_index()
      .rename(columns={"Offline": "Offline Intervals"})
)

offline_summary

,device_id,Offline Intervals
0,DVC-101,22
1,DVC-102,31
2,DVC-103,174
3,DVC-104,31
4,DVC-105,24


In [531]:
fig = px.bar(
    offline_summary,
    x="device_id",
    y="Offline Intervals",
    color="device_id",
    title="Offline Periods by Device",
    text_auto=True
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False
)

fig.show()
fig.write_html(
    "06_offline_periods.html",
    full_html=False,
    include_plotlyjs=True
)

In [532]:
flatline = (
    df.groupby("device_id")
      .agg(
          Min_Temp=("room_temp_c", "min"),
          Max_Temp=("room_temp_c", "max"),
          Std_Dev=("room_temp_c", "std")
      )
      .reset_index()
)

flatline

,device_id,Min_Temp,Max_Temp,Std_Dev
0,DVC-101,23.92,26.97,0.786770
1,DVC-102,23.92,27.42,0.815638
2,DVC-103,23.91,27.21,0.794056
3,DVC-104,23.50,27.19,1.139442
4,DVC-105,23.91,27.05,0.838013


In [533]:
power_threshold = (
    df["power_watt"].mean() +
    3 * df["power_watt"].std()
)

power_spikes = df[df["power_watt"] > power_threshold]

power_spikes.head()

,timestamp,device_id,room_temp_c,setpoint_c,ambient_temp_c,compressor_state,power_watt,mode,Offline,runtime_hours
14,2026-06-01 03:30:00,DVC-101,26.00,26.0,25.33,0.0,4803.8,OFF,False,0.00
52,2026-06-01 13:00:00,DVC-101,24.43,24.0,35.32,1.0,4273.5,COOL,False,0.25
190,2026-06-02 23:30:00,DVC-101,26.21,26.0,28.38,0.0,4692.4,OFF,False,0.00
280,2026-06-03 22:00:00,DVC-101,25.24,26.0,29.74,0.0,4046.1,OFF,False,0.00
476,2026-06-05 23:00:00,DVC-101,26.25,26.0,28.64,0.0,4214.5,OFF,False,0.00


In [534]:
print("Power Threshold:", power_threshold)
print("Number of Power Spikes:", len(power_spikes))

Power Threshold: 2483.3722105354154
Number of Power Spikes: 15


In [535]:
spike_summary = (
    power_spikes.groupby("device_id")
    .size()
    .reset_index(name="Power Spikes")
)

if spike_summary.empty:
    spike_summary = pd.DataFrame({
        "device_id": df["device_id"].unique(),
        "Power Spikes": 0
    })

spike_summary

,device_id,Power Spikes
0,DVC-101,15


In [536]:
fig = px.bar(
    spike_summary,
    x="device_id",
    y="Power Spikes",
    color="device_id",
    title="Abnormal Power Spikes by Device",
    text_auto=True
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False
)

fig.show()
fig.write_html(
    "07_power_spikes.html",
    full_html=False,
    include_plotlyjs=True
)

## Part 2 Summary & Observations

- Hourly and daily summaries showed that the HVAC devices maintained generally stable operating conditions throughout the monitoring period, with only minor variations in room temperature, power consumption, and compressor runtime.
- Room temperature trends indicated that most devices successfully maintained temperatures close to their configured setpoints, with normal fluctuations caused by compressor cycling and changes in operating conditions.
- Power consumption analysis revealed clear daily operating patterns, with peak energy usage occurring during daytime hours and significantly lower consumption during off-peak periods, reflecting typical HVAC operation based on occupancy and cooling demand.
- The anomaly detection analysis identified several notable issues. **DVC-103** experienced the highest number of offline intervals and a temporary reduction in power consumption, suggesting possible communication interruptions or temporary device downtime. **DVC-104** exhibited a prolonged flat-line room temperature despite changes in the setpoint, indicating a potential sensor or telemetry issue. **DVC-101** recorded multiple abnormal power spikes that were considerably higher than the normal operating range and may require further investigation.
- Overall, the HVAC systems demonstrated reliable performance and consistent energy usage across the observation period. While most devices operated within expected limits, the detected anomalies highlight specific devices that may benefit from preventive maintenance, sensor inspection, or closer operational monitoring.

In [537]:
df["energy_kwh"] = (df["power_watt"] * 0.25) / 1000
energy_summary = (
    df.groupby("device_id")
      .agg(
          Total_Energy_kWh=("energy_kwh", "sum")
      )
      .reset_index()
      .sort_values("Total_Energy_kWh", ascending=False)
)
energy_summary

,device_id,Total_Energy_kWh
0,DVC-101,179.567025
4,DVC-105,173.470575
1,DVC-102,166.787350
3,DVC-104,166.651175
2,DVC-103,143.578600


In [538]:
fig = px.bar(
    energy_summary,
    x="device_id",
    y="Total_Energy_kWh",
    color="device_id",
    title="Estimated Energy Consumption by Device",
    text_auto=".2f"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False,
    xaxis_title="HVAC Device",
    yaxis_title="Energy Consumption (kWh)"
)

fig.show()
fig.write_html(
    "08_energy_consumption.html",
    full_html=False,
    include_plotlyjs=True
)

In [539]:
df["temp_difference"] = abs(df["room_temp_c"] - df["setpoint_c"])
df["setpoint_miss"] = df["temp_difference"] > 1

setpoint_summary = (
    df.groupby("device_id")
      .agg(
          Setpoint_Misses=("setpoint_miss", "sum")
      )
      .reset_index()
      .sort_values("Setpoint_Misses", ascending=False)
)

setpoint_summary

,device_id,Setpoint_Misses
3,DVC-104,335
2,DVC-103,101
4,DVC-105,99
0,DVC-101,98
1,DVC-102,95


In [540]:
fig = px.bar(
    setpoint_summary,
    x="device_id",
    y="Setpoint_Misses",
    color="device_id",
    title="Setpoint Misses by Device",
    text_auto=True
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False,
    xaxis_title="HVAC Device",
    yaxis_title="Number of Setpoint Misses"
)

fig.show()
fig.write_html(
    "09_setpoint_misses.html",
    full_html=False,
    include_plotlyjs=True
)

### Key Observations

- The analysis shows noticeable differences in energy consumption across the HVAC devices. Devices with higher energy usage generally exhibited longer compressor runtime and higher average power consumption, making them the primary contributors to overall energy usage.

- Ranking the devices by both energy consumption and setpoint misses highlights variations in operational efficiency. Devices with frequent setpoint misses may require additional energy to maintain the desired temperature, indicating potential control, sensor, or maintenance issues.

- Combining energy usage with temperature control performance provides a practical basis for prioritizing maintenance and optimization efforts. Focusing on high-energy devices and those with frequent setpoint deviations can improve system efficiency, reduce operating costs, and enhance occupant comfort.